# Mass Dataset Processing Pipeline

Este notebook descarga, descomprime y procesa múltiples datasets de ajedrez de forma secuencial.

**Pipeline por dataset:**
1. Descarga archivo .pgn.zst (~30 GB)
2. Descomprime a .pgn (~200 GB) [~1-2 horas]
3. Procesa con pipeline de estilometría [~4.5 horas]
4. Genera PGNs por jugador (sin imágenes)
5. Limpia archivos temporales
6. Procesa siguiente dataset

In [9]:
import sys
import os
from pathlib import Path
import subprocess
import time
from datetime import datetime, timedelta
import requests
from tqdm import tqdm
import logging

# Añadir directorio labs al path
labs_dir = Path.cwd().parent
if str(labs_dir) not in sys.path:
    sys.path.insert(0, str(labs_dir))

# Configurar logging persistente
log_file = labs_dir / "mass_dataset_processing.log"
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(log_file, mode='a'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

logger.info("="*70)
logger.info(f"SESSION START: {datetime.now().isoformat()}")
logger.info("="*70)
logger.info(f"Working directory: {Path.cwd()}")
logger.info(f"Labs directory: {labs_dir}")
logger.info(f"Log file: {log_file}")

print(f"Working directory: {Path.cwd()}")
print(f"Labs directory: {labs_dir}")
print(f"Log file: {log_file}")

2025-12-17 13:21:29,552 - INFO - ======================================================================
2025-12-17 13:21:29,554 - INFO - SESSION START: 2025-12-17T13:21:29.554483
2025-12-17 13:21:29,555 - INFO - ======================================================================
2025-12-17 13:21:29,557 - INFO - Working directory: /home/andrewyernau/dev/jupyter/labs/notebooks
2025-12-17 13:21:29,558 - INFO - Labs directory: /home/andrewyernau/dev/jupyter/labs
2025-12-17 13:21:29,559 - INFO - Log file: /home/andrewyernau/dev/jupyter/labs/mass_dataset_processing.log


Working directory: /home/andrewyernau/dev/jupyter/labs/notebooks
Labs directory: /home/andrewyernau/dev/jupyter/labs
Log file: /home/andrewyernau/dev/jupyter/labs/mass_dataset_processing.log


## Configuración

## Monitorización del progreso

Ejecuta esta celda para ver el estado actual del procesamiento (incluso si cerraste el navegador):

In [10]:
# Ver las últimas líneas del log para monitorear progreso
log_file = labs_dir / "mass_dataset_processing.log"

if log_file.exists():
    print(f"Estado del procesamiento (log: {log_file})\n")
    print("="*70)
    
    # Leer últimas 50 líneas
    with open(log_file, 'r') as f:
        lines = f.readlines()
        last_lines = lines[-50:] if len(lines) > 50 else lines
        
    for line in last_lines:
        print(line.rstrip())
    
    print("\n" + "="*70)
    print(f"Total líneas en log: {len(lines)}")
    print(f"Última actualización: {datetime.fromtimestamp(log_file.stat().st_mtime)}")
else:
    print(f"⚠️  Log file no encontrado: {log_file}")
    print("El procesamiento aún no ha iniciado o el archivo fue eliminado.")

Estado del procesamiento (log: /home/andrewyernau/dev/jupyter/labs/mass_dataset_processing.log)

2025-12-17 08:31:39,683 - ERROR - No such comm target registered: jupyter.widget.control
2025-12-17 08:31:39,686 - WARNING - No such comm: 22038ee2-952e-447d-8bd4-273eb64505b1
2025-12-17 08:31:58,497 - INFO - ======================================================================
2025-12-17 08:31:58,498 - INFO - INICIO DEL PROCESAMIENTO MASIVO
2025-12-17 08:31:58,500 - INFO - ======================================================================
2025-12-17 08:31:58,501 - INFO - Total de datasets: 1
2025-12-17 08:31:58,502 - INFO - Hora de inicio: 2025-12-17 08:31:58
2025-12-17 08:31:58,504 - INFO - ETA estimado: ~6 horas (6h por dataset)
2025-12-17 08:31:58,506 - INFO - ######################################################################
2025-12-17 08:31:58,507 - INFO - # DATASET 1/1: lichess_db_standard_rated_2025-08.pgn
2025-12-17 08:31:58,509 - INFO - ###################################

In [11]:
# Lista de URLs de datasets a procesar
DATASET_URLS = [
    #"https://database.lichess.org/standard/lichess_db_standard_rated_2025-11.pgn.zst",
    #"https://database.lichess.org/standard/lichess_db_standard_rated_2025-10.pgn.zst",
    "https://database.lichess.org/standard/lichess_db_standard_rated_2025-08.pgn.zst",
    #"https://database.lichess.org/standard/lichess_db_standard_rated_2025-07.pgn.zst",
    #"https://database.lichess.org/standard/lichess_db_standard_rated_2025-06.pgn.zst",
    #"https://database.lichess.org/standard/lichess_db_standard_rated_2025-05.pgn.zst",
    #"https://database.lichess.org/standard/lichess_db_standard_rated_2025-04.pgn.zst",
    #"https://database.lichess.org/standard/lichess_db_standard_rated_2025-03.pgn.zst",
    #"https://database.lichess.org/standard/lichess_db_standard_rated_2025-02.pgn.zst",
    #"https://database.lichess.org/standard/lichess_db_standard_rated_2025-01.pgn.zst",
]

# Directorios
DOWNLOAD_DIR = Path("/tmp/chess_downloads")  # Directorio temporal para descargas
OUTPUT_BASE = labs_dir / "dataset" / "mass_processed"  # Output final

# Parámetros del pipeline
EVENT_TYPE = "Rated Blitz game"  # Tipo de evento a filtrar
NUM_PLAYERS = 600  # Número de jugadores a extraer (0 = todos los que cumplan requisitos)
GAMES_PER_PLAYER = 50  # Partidas objetivo por jugador
MIN_THRESHOLD = 0.7  # Umbral mínimo (proporción de partidas válidas)
MOVE_START = 15  # Jugada inicial
MOVE_END = 30  # Jugada final
NUM_BLOCKS = 3  # Número de bloques temporales

# Crear directorios
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_BASE.mkdir(parents=True, exist_ok=True)

print(f"Download directory: {DOWNLOAD_DIR}")
print(f"Output directory: {OUTPUT_BASE}")
print(f"\nPipeline config:")
print(f"  Event type: {EVENT_TYPE}")
print(f"  Players: {NUM_PLAYERS}")
print(f"  Games per player: {GAMES_PER_PLAYER}")
print(f"  Move range: {MOVE_START}-{MOVE_END}")
print(f"  Blocks: {NUM_BLOCKS}")

Download directory: /tmp/chess_downloads
Output directory: /home/andrewyernau/dev/jupyter/labs/dataset/mass_processed

Pipeline config:
  Event type: Rated Blitz game
  Players: 600
  Games per player: 50
  Move range: 15-30
  Blocks: 3


In [12]:
def cleanup_files(*files):
    """
    Elimina archivos temporales de forma segura.
    
    Parameters
    ----------
    *files : Path
        Rutas de archivos a eliminar
    """
    print(f"\n{'='*70}")
    print("Limpiando archivos temporales")
    print(f"{'='*70}\n")
    
    for file in files:
        if isinstance(file, Path) and file.exists():
            try:
                size_gb = file.stat().st_size / (1024**3)
                file.unlink()
                print(f"  ✓ Eliminado: {file.name} ({size_gb:.2f} GB)")
                logger.info(f"Archivo eliminado: {file} ({size_gb:.2f} GB)")
            except Exception as e:
                print(f"  ✗ Error eliminando {file}: {e}")
                logger.error(f"Error eliminando {file}: {e}")
        else:
            print(f"  (no existe: {file})")


def format_eta(seconds: float) -> str:
    """
    Formatea segundos a formato legible horas:minutos.
    
    Parameters
    ----------
    seconds : float
        Segundos a formatear
    
    Returns
    -------
    str
        Tiempo formateado como "Xh Ym"
    """
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    return f"{hours}h {minutes}m"


def download_file(url: str, destination: Path) -> bool:
    """
    Descarga un archivo con barra de progreso.
    
    Parameters
    ----------
    url : str
        URL del archivo
    destination : Path
        Ruta de destino
    
    Returns
    -------
    bool
        True si la descarga fue exitosa
    """
    try:
        print(f"\nDescargando: {url}")
        print(f"Destino: {destination}")
        
        response = requests.get(url, stream=True)
        response.raise_for_status()
        
        total_size = int(response.headers.get('content-length', 0))
        
        with open(destination, 'wb') as f:
            with tqdm(total=total_size, unit='B', unit_scale=True, desc=destination.name) as pbar:
                for chunk in response.iter_content(chunk_size=8192):
                    if chunk:
                        f.write(chunk)
                        pbar.update(len(chunk))
        
        print(f"✓ Descarga completada: {destination}")
        return True
    
    except Exception as e:
        print(f"✗ Error descargando {url}: {e}")
        if destination.exists():
            destination.unlink()
        return False


def decompress_zst(compressed_file: Path, output_file: Path) -> bool:
    """
    Descomprime un archivo .zst usando zstd.
    
    Parameters
    ----------
    compressed_file : Path
        Archivo comprimido .zst
    output_file : Path
        Archivo de salida descomprimido
    
    Returns
    -------
    bool
        True si la descompresión fue exitosa
    """
    try:
        print(f"\nDescomprimiendo: {compressed_file}")
        print(f"Destino: {output_file}")
        
        start_time = time.time()
        
        # Usar zstd para descomprimir (--force para sobrescribir si existe)
        cmd = ["zstd", "-d", str(compressed_file), "-o", str(output_file), "--long=31", "--force"]
        result = subprocess.run(cmd, check=True, capture_output=True, text=True)
        
        elapsed = time.time() - start_time
        print(f"✓ Descompresión completada en {elapsed/60:.1f} minutos")
        print(f"  Tamaño final: {output_file.stat().st_size / (1024**3):.2f} GB")
        
        return True
    
    except subprocess.CalledProcessError as e:
        print(f"✗ Error descomprimiendo {compressed_file}: {e}")
        print(f"  stderr: {e.stderr}")
        if output_file.exists():
            output_file.unlink()
        return False
    
    except Exception as e:
        print(f"✗ Error inesperado: {e}")
        if output_file.exists():
            output_file.unlink()
        return False


def run_pipeline(pgn_file: Path, output_dir: Path, dataset_name: str) -> bool:
    """
    Ejecuta el pipeline de estilometría SOLO para generar PGNs (sin imágenes).
    
    Parameters
    ----------
    pgn_file : Path
        Archivo PGN a procesar
    output_dir : Path
        Directorio de salida
    dataset_name : str
        Nombre del dataset (para logging)
    
    Returns
    -------
    bool
        True si el procesamiento fue exitoso
    """
    try:
        print(f"\n{'='*70}")
        print(f"Procesando pipeline: {dataset_name}")
        print(f"{'='*70}\n")
        
        # Importar usando el mismo path que usa 002_blocks_siamese_cnn.ipynb
        import sys
        from pathlib import Path
        obsolete_path = labs_dir / "obsolete_v001" / "scripts"
        if str(obsolete_path) not in sys.path:
            sys.path.insert(0, str(obsolete_path))
        
        from extract_player_games_by_event_parallel import ParallelPlayerExtractorByEvent
        
        start_time = time.time()
        
        # Estructura de salida
        event_safe = EVENT_TYPE.replace(" ", "_").replace("/", "_")
        event_dir = output_dir / "events" / event_safe
        player_pgns_dir = event_dir / "player_pgns"
        
        # FASE 1: Extraer jugadores y generar PGNs
        print(f"Extrayendo jugadores de: {pgn_file}")
        print(f"Output: {player_pgns_dir}\n")
        
        
        extractor = ParallelPlayerExtractorByEvent(
            pgn_path=pgn_file,
            event_type=EVENT_TYPE,
            num_players=NUM_PLAYERS,
            games_per_player=GAMES_PER_PLAYER,
            min_games_threshold=MIN_THRESHOLD
        )
        
        valid_players = extractor.extract_games()
        
        if not valid_players:
            print("ERROR: No se encontraron jugadores con suficientes partidas")
            return False
        
        extractor.save_player_games(player_pgns_dir, valid_players)

        # Mostrar estadísticas
        elapsed = time.time() - start_time
        print(f"\n{'='*70}")
        print(f"✓ Pipeline completado exitosamente")
        print(f"{'='*70}")
        print(f"Jugadores procesados: {len(valid_players)}")
        print(f"Tiempo total: {elapsed/60:.1f} minutos")
        print(f"Output: {player_pgns_dir}")
        
        logger.info(f"Pipeline {dataset_name} completado: {len(valid_players)} jugadores en {elapsed/60:.1f}min")
        
        return True
    
    except Exception as e:
        print(f"\n✗ Error en pipeline: {e}")
        logger.error(f"Error en pipeline {dataset_name}: {e}")
        import traceback
        traceback.print_exc()
        return False


In [ ]:
#Estadísticas globales
global_stats = { 'total_datasets': len(DATASET_URLS), 'processed': 0, 'failed': 0, 'start_time': time.time(), 'results': [] }

logger.info("="*70)
logger.info("INICIO DEL PROCESAMIENTO MASIVO")
logger.info("="*70)
logger.info(f"Total de datasets: {len(DATASET_URLS)}") 
logger.info(f"Hora de inicio: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}") 
logger.info(f"ETA estimado: ~{len(DATASET_URLS) * 6} horas (6h por dataset)")

print(f"\n{'='*70}") 
print(f"INICIO DEL PROCESAMIENTO MASIVO") 
print(f"{'='*70}\n") 
print(f"Total de datasets: {len(DATASET_URLS)}") 
print(f"Hora de inicio: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}") 
print(f"\nETA estimado: ~{len(DATASET_URLS) * 6} horas (6h por dataset)\n")

for idx, url in enumerate(DATASET_URLS, 1):
    dataset_start = time.time()
    # Construir nombres de archivo correctamente
    # URL: https://.../lichess_db_standard_rated_2025-08.pgn.zst
    # compressed_file: lichess_db_standard_rated_2025-08.pgn.zst
    # decompressed_file: lichess_db_standard_rated_2025-08.pgn
    # dataset_name (para logs): lichess_db_standard_rated_2025-08
    compressed_file = DOWNLOAD_DIR / Path(url).name
    decompressed_file = DOWNLOAD_DIR / Path(url).name.replace('.zst', '')
    dataset_name = decompressed_file.stem  # Para logs, sin extensión

logger.info("#"*70)
logger.info(f"# DATASET {idx}/{len(DATASET_URLS)}: {dataset_name}")
logger.info("#"*70)

print(f"\n\n{'#'*70}")
print(f"# DATASET {idx}/{len(DATASET_URLS)}: {dataset_name}")
print(f"{'#'*70}\n")

success = True
error_msg = None

try:
    # Verificar si ya existe archivo descomprimido
    if decompressed_file.exists():
        logger.info(f"✓ Archivo descomprimido ya existe: {decompressed_file}")
        print(f"\n✓ Archivo descomprimido encontrado: {decompressed_file.name}")
        file_size_gb = decompressed_file.stat().st_size / (1024**3)
        print(f"  Tamaño: {file_size_gb:.2f} GB")
        logger.info(f"Saltando descarga y descompresión (archivo: {file_size_gb:.2f} GB)")
        compressed_file = None  # No hay archivo comprimido a limpiar
    else:
        # PASO 1: Descargar
        if not compressed_file.exists():
            logger.info(f"PASO 1/4: Descargando {url}")
            if not download_file(url, compressed_file):
                success = False
                error_msg = "Descarga fallida"
                raise Exception(error_msg)
        else:
            logger.info(f"✓ Ya descargado: {compressed_file}")
            print(f"✓ Ya descargado: {compressed_file}")
        
        # PASO 2: Descomprimir
        logger.info(f"PASO 2/4: Descomprimiendo {compressed_file}")
        if not decompress_zst(compressed_file, decompressed_file):
            success = False
            error_msg = "Descompresión fallida"
            raise Exception(error_msg)
        
        # PASO 2.5: Borrar archivo comprimido INMEDIATAMENTE
        logger.info(f"PASO 2.5/4: Borrando archivo comprimido")
        cleanup_files(compressed_file)
    
    # PASO 3: Procesar pipeline (solo PGNs)
    logger.info(f"PASO 3/4: Procesando pipeline")
    if not run_pipeline(decompressed_file, OUTPUT_BASE, dataset_name):
        success = False
        error_msg = "Pipeline fallido"
        raise Exception(error_msg)
    
    # PASO 4: Borrar archivo descomprimido INMEDIATAMENTE
    logger.info(f"PASO 4/4: Borrando archivo descomprimido")
    cleanup_files(decompressed_file)
    
except Exception as e:
    logger.error(f"✗ Error procesando {dataset_name}: {e}")
    print(f"\n✗ Error procesando {dataset_name}: {e}")
    success = False
    if not error_msg:
        error_msg = str(e)
    # Intentar limpiar archivos incluso si hubo error
    logger.info("Limpiando archivos después del error...")
    files_to_clean = [decompressed_file]
    if compressed_file and compressed_file.exists():
        files_to_clean.append(compressed_file)
    cleanup_files(*files_to_clean)

# Actualizar estadísticas
dataset_elapsed = time.time() - dataset_start

if success:
    global_stats['processed'] += 1
    status = "✓ ÉXITO"
else:
    global_stats['failed'] += 1
    status = "✗ FALLO"

global_stats['results'].append({
    'dataset': dataset_name,
    'success': success,
    'time': dataset_elapsed,
    'error': error_msg
})

# Mostrar progreso
total_elapsed = time.time() - global_stats['start_time']
avg_time = total_elapsed / idx
remaining = len(DATASET_URLS) - idx
eta = remaining * avg_time

logger.info("="*70)
logger.info(f"RESULTADO DATASET {idx}/{len(DATASET_URLS)}: {status}")
logger.info("="*70)
logger.info(f"Tiempo dataset: {dataset_elapsed/3600:.2f}h")
logger.info(f"Tiempo acumulado: {total_elapsed/3600:.2f}h")
logger.info(f"Procesados: {global_stats['processed']} | Fallados: {global_stats['failed']}")
logger.info(f"ETA restante: {format_eta(eta)}")
logger.info("="*70)

print(f"\n{'='*70}")
print(f"RESULTADO DATASET {idx}/{len(DATASET_URLS)}: {status}")
print(f"{'='*70}")
print(f"Tiempo dataset: {dataset_elapsed/3600:.2f}h")
print(f"Tiempo acumulado: {total_elapsed/3600:.2f}h")
print(f"Procesados: {global_stats['processed']} | Fallados: {global_stats['failed']}")
print(f"ETA restante: {format_eta(eta)}")
print(f"{'='*70}\n")

#Resumen final
total_time = time.time() - global_stats['start_time']

logger.info("="*70) 
logger.info("PROCESAMIENTO MASIVO COMPLETADO") 
logger.info("="*70) 
logger.info(f"Total datasets: {global_stats['total_datasets']}") 
logger.info(f"Procesados exitosamente: {global_stats['processed']}") 
logger.info(f"Fallados: {global_stats['failed']}") 
logger.info(f"Tiempo total: {total_time/3600:.2f} horas") 
logger.info("Resultados por dataset:")

print(f"\n\n{'='*70}") 
print(f"PROCESAMIENTO MASIVO COMPLETADO") 
print(f"{'='*70}\n") 
print(f"Total datasets: {global_stats['total_datasets']}") 
print(f"Procesados exitosamente: {global_stats['processed']}") 
print(f"Fallados: {global_stats['failed']}")
print(f"Tiempo total: {total_time/3600:.2f} horas") 
print(f"\nResultados por dataset:\n")

for result in global_stats['results']:
    status = "✓" if result['success'] else "✗"
    time_str = f"{result['time'] / 3600:.2f}h"
    error_str = f" ({result['error']})" if result['error'] else ""

    log_line = f" {status} {result['dataset']:<40} {time_str:>8}{error_str}"
    logger.info(log_line)
    print(log_line)

logger.info(f"Output final: {OUTPUT_BASE}")
logger.info("=" * 70)
print(f"\nOutput final: {OUTPUT_BASE}")
print(f"{'=' * 70}\n")


2025-12-17 13:21:40,285 - INFO - ======================================================================
2025-12-17 13:21:40,286 - INFO - INICIO DEL PROCESAMIENTO MASIVO
2025-12-17 13:21:40,287 - INFO - ======================================================================
2025-12-17 13:21:40,288 - INFO - Total de datasets: 1
2025-12-17 13:21:40,289 - INFO - Hora de inicio: 2025-12-17 13:21:40
2025-12-17 13:21:40,290 - INFO - ETA estimado: ~6 horas (6h por dataset)
2025-12-17 13:21:40,292 - INFO - ######################################################################
2025-12-17 13:21:40,294 - INFO - # DATASET 1/1: lichess_db_standard_rated_2025-08
2025-12-17 13:21:40,295 - INFO - ######################################################################
2025-12-17 13:21:40,298 - INFO - ✓ Archivo descomprimido ya existe: /tmp/chess_downloads/lichess_db_standard_rated_2025-08.pgn
2025-12-17 13:21:40,300 - INFO - Saltando descarga y descompresión (archivo: 201.11 GB)
2025-12-17 13:21:40,301 - 


INICIO DEL PROCESAMIENTO MASIVO

Total de datasets: 1
Hora de inicio: 2025-12-17 13:21:40

ETA estimado: ~6 horas (6h por dataset)



######################################################################
# DATASET 1/1: lichess_db_standard_rated_2025-08
######################################################################


✓ Archivo descomprimido encontrado: lichess_db_standard_rated_2025-08.pgn
  Tamaño: 201.11 GB

Procesando pipeline: lichess_db_standard_rated_2025-08

Extrayendo jugadores de: /tmp/chess_downloads/lichess_db_standard_rated_2025-08.pgn
Output: /home/andrewyernau/dev/jupyter/labs/dataset/mass_processed/events/Rated_Blitz_game/player_pgns


EXTRACCIÓN POR EVENTO
Evento: Rated Blitz game
Objetivo: 600 jugadores
Partidas/jugador: 50
Mínimo: 35 partidas (70%)
Workers: 72 (cores: 72)

FASE 1: Descubrimiento de jugadores...
  Procesando 72 chunks en paralelo...
    Chunk 203074MB: 10,000 partidas (21.0/2860.1 MB, 4,185 del evento)
    Chunk 194494MB: 10,000 partidas (22.0

In [5]:
# Verificar que zstd está instalado
try:
    subprocess.run(["zstd", "--version"], check=True, capture_output=True)
    print("✓ zstd está instalado")
except FileNotFoundError:
    print("✗ zstd no está instalado. Instalar con: sudo apt-get install zstd")
    raise

# Verificar lista de URLs
if not DATASET_URLS:
    print("⚠️  ADVERTENCIA: La lista DATASET_URLS está vacía")
    print("   Añade URLs de datasets en la celda de configuración")
else:
    print(f"✓ {len(DATASET_URLS)} datasets en la cola\n")

✓ zstd está instalado
✓ 11 datasets en la cola



## Inspección de resultados

In [6]:
# Ver estructura de salida
import os

print(f"Estructura de salida:\n")
for root, dirs, files in os.walk(OUTPUT_BASE):
    level = root.replace(str(OUTPUT_BASE), '').count(os.sep)
    indent = ' ' * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = ' ' * 2 * (level + 1)
    for file in files[:5]:  # Mostrar solo primeros 5 archivos
        print(f"{subindent}{file}")
    if len(files) > 5:
        print(f"{subindent}... ({len(files)-5} archivos más)")

Estructura de salida:

mass_processed/


In [7]:
# Contar PGNs generados por evento
events_dir = OUTPUT_BASE / "events"

if events_dir.exists():
    print("PGNs generados por evento:\n")
    for event_dir in sorted(events_dir.iterdir()):
        if event_dir.is_dir():
            pgns_dir = event_dir / "player_pgns"
            if pgns_dir.exists():
                pgn_files = list(pgns_dir.glob("*.pgn"))
                print(f"  {event_dir.name}: {len(pgn_files)} jugadores")
else:
    print("No se encontraron eventos procesados")

No se encontraron eventos procesados


## Notas

- Los archivos temporales (.zst y .pgn descomprimidos) se eliminan automáticamente después de procesar cada dataset
- Solo se generan PGNs por jugador (no imágenes) para ahorrar espacio y tiempo
- Los PGNs generados se pueden usar posteriormente para generar imágenes con otros notebooks
- El tiempo estimado por dataset es ~6 horas (2h descarga+descompresión + 4.5h procesamiento)
- Si un dataset falla, el pipeline continúa con el siguiente